# Reverse Engineering an ASIC

Download the required dependencies.
```bash
%pip install -r requirements.txt ipykernel
```
 Graphviz is optional for extraction and simulation, but it is used throughout this notebook to render `.dot` graphs. Install it by running this command in your terminal (if you already use homebrew)
 ```bash
 brew install graphviz
 ```
 Alternatively, you could upload the dot files to any online .dot viewer.

## 1. Getting Started

GDS files are stored in binary, so we can't just scroll through. Lets open the file in a layout viewer such as KLayout, or an online gds viewer such as [tinytapeout](https://gds-viewer.tinytapeout.com/). A screenshot from tinytapeout is attached here:

<img src="artifacts/images/puzzlegds_viz.png" alt="Visualized puzzle.gds" width="400">

Certainly looks interesting, but not too interpretable. Zooming in reveals metal wires, vias (joining wires across layers), and individual transistors. Fortunately, the file stores objects at the granularity of cell types: it places instances of SKY130 cells such as AND gates, multiplexers, and flip-flops at specified coordiantes. The definitions of all the cell types can be found in the [SKY130 PDK](https://skywater-pdk.readthedocs.io/en/main/contents/libraries.html).

The first plan of attack is then to just see what cell types this gds file contains:

In [6]:
# Compact view of: python -m tools.inspect_gds puzzle.gds
from collections import Counter

import gdstk

from tools.inspect_gds import classify

lib = gdstk.read_gds('puzzle.gds')
top = lib.top_level()[0]
counts = Counter(ref.cell.name for ref in top.references)
print(f'Unique cell types: {len(counts)}')
print(f'Instances: {len(top.references)}')
labels = list(dict.fromkeys(lab.text for lab in top.labels))
print(f'Top-level labels: {labels}')
print()
print('A few cell types and occurring frequencies:')
for name in (
    'VIA_M1M2_PR',
    'sky130_fd_sc_hd__dfrtp_2',
    'sky130_fd_sc_hd__nor2_2',
    'sky130_fd_sc_hd__nand2_2',
    'sky130_fd_sc_hd__inv_2',
    'INTERNAL_3',
    'INTERNAL_7',
):
    print(f'  {counts[name]:5}  {name}  [{classify(name)}]')


Unique cell types: 80
Instances: 9875
Top-level labels: ['I', 'O[0]', 'O[1]', 'O[2]', 'O[3]', 'O[4]', 'O[5]', 'O[6]', 'O[7]', 'clk', 'enable', 'rst_n', 'success', 'VGND', 'VPWR']

A few cell types and occurring frequencies:
   3154  VIA_M1M2_PR  [via]
     84  sky130_fd_sc_hd__dfrtp_2  [logic]
     49  sky130_fd_sc_hd__nor2_2  [logic]
     39  sky130_fd_sc_hd__nand2_2  [logic]
     25  sky130_fd_sc_hd__inv_2  [logic]
     21  INTERNAL_3  [other]
     15  INTERNAL_7  [other]


We now know the identity of all the gates (for example, nor2, and2, etc.), the wires, vias, and the inputs and outputs to the circuit as a whole: `clk`, `rst_n`, `enable`, `I`, `O[7:0]`, and `success`. 

There appear to be some unidentified cell types as well, 21 `INTERNAL_3` instances and 15 `INTERNAL_7`. Viewing them in a gds viewer (the below screenshot is from Klayout) reveals that they are a morse code encoding of the phrase "PER ARENAM AD ASTRA".

<img src="artifacts/images/morse.jpg" alt="Visualized puzzle.gds" width="800">

We next need to find out which gates are connected to which.

## 2. Connectivity

The GDS file gives us the location of the pins of each gate and all metal conductors. On each layer, we merge touching wires into connected regions. Then we merge regions on adjacent layers wherever a via connects them, into one giant conducting net. Next, go through every pin on every gate and find out which net it touches, and assemble into a big list of nets and pins.

In [12]:
# Extract nets from the gds. This can take a little while.
!.venv/bin/python -m tools.extract_netlist puzzle.gds --json generated/puzzle.json

top cell: puzzle
merging conductor layers...
  li1   raw=13931  merged regions=6680
  met1  raw=14366  merged regions=3001
  met2  raw= 8517  merged regions=2060
  met3  raw= 2547  merged regions=811
  met4  raw=  855  merged regions=45
  met5  raw=  144  merged regions=18
stitching layers through vias...
  li1->met1: 19764 cuts, 19764 stitched
  met1->met2: 6869 cuts, 6869 stitched
  met2->met3: 3423 cuts, 3423 stitched
  met3->met4: 3159 cuts, 3159 stitched
  met4->met5: 108 cuts, 108 stitched
assigning pins to nets...

=== 741 nets extracted ===


In [7]:
# Look at a few extracted nets, skip over the massive VGND and VPWR nets, they just provide power.
import json
from pathlib import Path

ListOfNets = Path('generated/puzzle.json')
if not ListOfNets.exists():
    ListOfNets = Path('artifacts/netlists/puzzle.json')

raw = json.loads(ListOfNets.read_text())
for name in ('n0004', 'n0008', 'n0012'):
    print(name, json.dumps(raw['nets'][name], indent=2))


n0004 {
  "aliases": [],
  "terminals": [
    "U100_dfrtp_2.Q",
    "U103_a31o_2.B1",
    "U147_and2b_2.A_N",
    "U149_nand4_2.C"
  ]
}
n0008 {
  "aliases": [],
  "terminals": [
    "U102_a31o_2.B1",
    "U154_nand4_2.C",
    "U155_and2b_2.A_N",
    "U98_dfrtp_2.Q"
  ]
}
n0012 {
  "aliases": [],
  "terminals": [
    "U104_o21a_2.A1",
    "U105_nand2b_2.A_N",
    "U147_and2b_2.B",
    "U172_dfrtp_2.Q"
  ]
}


A record such as `U119_clkbuf_4.A` means pin `A` of a particular clock-buffer instance touches that net. Cell model definitions from the official SKY130 Liberty data tell us pin directions and the Boolean functions they implement. If all is well, each net should have a single pin driving it, and all other pins receiving that as input. We can form a directed bipartite graph: gate outputs drive net nodes, and net nodes fan out to gate inputs - while sanity checking that each net have exactly one driver.

In [4]:
!.venv/bin/python -m tools.analyze_netlist {ListOfNets} summary

Source:          generated/puzzle.json
Instances:       728
Nets:            741 (739 non-power)
Terminals:       4223
State elements:  92

Ports (direction inferred from cell pin directions):
  I                input             drivers=0  loads=45
  O[0]             output            drivers=1  loads=0
  O[1]             output            drivers=1  loads=0
  O[2]             output            drivers=1  loads=0
  O[3]             output            drivers=1  loads=0
  O[4]             output            drivers=1  loads=0
  O[5]             output            drivers=1  loads=0
  O[6]             output            drivers=1  loads=0
  O[7]             output            drivers=1  loads=0
  VGND             power             drivers=0  loads=0
  VPWR             power             drivers=0  loads=0
  clk              input             drivers=0  loads=1
  enable           input             drivers=0  loads=1
  rst_n            input             drivers=0  loads=88
  success          ou

So we have 728 logical instances, 741 nets, 92 memory elements (flip-flops), and one suspicious floating internal net (no driver), `n0550`...to be investigated later.

### Lets just draw everything

We have the whole connectivity graph...lets render it out. One subtlety: to keep things streamlined, we recursively work backwards from our ultimate goal (making success = 1), and render only those gates that could possibly influence the value of success - the "backwards cone" influencing the success net. The code to do this is in `tools/analyze_netlist.py`.

<img src="graphviz.svg" alt="Backwards cone of success" width="950">

If you want to render this yourself, the code is provided in the next cell. There's no way we're making sense of this with the eyeball test. A tiny "success" node is visible at the top right. We must smartly query this graph in small sections to try and infer its function.

In [5]:
# Reproduce the  graph shown above.
!python -m tools.analyze_netlist {ListOfNets} cone success --through-flops --dot generated/puzzle-success-backwards-cone.dot
!dot -Tsvg generated/puzzle-success-backwards-cone.dot -o generated/puzzle-success-backwards-cone.svg

zsh:1: command not found: python


## 3. Rehearse the method on the warm-up

Before interpreting 92 state bits, we'll test the pipeline on something small. The warm-up contains two eight-bit enabled shift registers. Once those are recognized, `success` is a function of only the 16 register bits. Enumerating all `2^16 = 65,536` assignments is pretty reasonable.

In [25]:
!python -m tools.analyze_netlist artifacts/netlists/warmup.json shift-registers
!python -m examples.warmup_exhaustive

Number of shift registers: 2

shift_A: ShiftRegister[8]
  serial input: A
  enable:       en
  clock:        clk
  members:      8 mux, 8 flip-flops

shift_B: ShiftRegister[8]
  serial input: B
  enable:       en
  clock:        clk
  members:      8 mux, 8 flip-flops

evaluated states: 65536
successful states: 15
successful pairs: [(241, 255), (242, 254), (243, 253), (244, 252), (245, 251), (246, 250), (247, 249), (248, 248), (249, 247), (250, 246), (251, 245), (252, 244), (253, 243), (254, 242), (255, 241)]
distinct A+B values: [496]


The successful states all satisfy `A + B = 496`.

Trying the same trick on the main puzzle would mean `2^92` states to enumerate - way too many. The warm-up exercise does, however, give us the idea to look for and abstract away shift registers in the main puzzle into their own block - gathering up clutter and making it interpretable.

## 4. Take some hints - the sample inputs file

It is time to analyze the provided `example_inputs.vcd` file. It instructs us to look at it through a waveform viewer, we can use an [online vcd viewer](https://app.surfer-project.org/). A biref section of it looks like this:

<img src="artifacts/images/vcd_view.png" alt="VCD view" width="800">

Success is always 0, but on application of this particular input, and setting the enable to 0, results in some output. With some educated guesswork, we can try seeing what ASCII characters this output corresponds to, and we find that it says "TRY AGAIN". Which (and how many) input bits did we feed in the first place?

In [8]:
from tools.helpers.vcd import inputs_at_rising_clock_edge

samples = inputs_at_rising_clock_edge('example_inputs.vcd')
attempts, current = [], []
for sample in samples:
    if sample['enable']:
        current.append(bool(sample['I']))
    elif current:
        attempts.append(current)
        current = []
if current:
    attempts.append(current)

print('Input string length for each "attempt":', [len(bits) for bits in attempts])

for index, bits in enumerate(attempts):
    print("Input ", index, ":")
    print(''.join('1' if bit else '0' for bit in bits))

Input string length for each "attempt": [121, 121]
Input  0 :
0010101000000010110000101001100000000010000001110110000100101100001110011000000010110000001011100000000010000011001110000
Input  1 :
1101011000010011110000000001000001000011000011101110000100001100001001011000000101110000110011100000000010000000000100000


Okay! Each input attempt appears to be 121 bits long. The actual inputs bits are also dumped into the output, can we possibly make sense of them? Lets try 11 groups of 11, and convert the 11-bit long strings into a decimal number. With a little trial and error, turns out they are ASCII characters encoded little-endian style (the first bit into the circuit is the most significant bit). Lets decode it:

In [9]:
def decode_eleven_bit_words(bits):
    values = [
        sum(bit << offset for offset, bit in enumerate(bits[start:start + 11]))
        for start in range(0, len(bits), 11)
    ]
    return values, ''.join(chr(value) for value in values)

for index, bits in enumerate(attempts, 1):
    values, text = decode_eleven_bit_words(bits)
    print(index, values, repr(text))

1 [84, 104, 101, 32, 110, 105, 103, 104, 116, 32, 115] 'The night s'
2 [107, 121, 32, 97, 119, 97, 105, 116, 115, 32, 32] 'ky awaits  '


**“The night sky awaits”**! 
Great. Now, a natural thing to do is to treat this like a black box and write code that would allow us to send in a 121 bit word, simulate the whole circuit, and see what the value of success and what the outputs are.

## 5. Simulate

We want a function that takes a 121-bit word, clocks it into the chip, and tells us what `success` and `O[7:0]` do.

The circuit is synchronous: on a rising clock edge every flip-flop samples its D pin, and all Qs update together. Gates in between are just the boolean functions from the SKY130 cell library. So a "simulation" is: remember the current Qs, evaluate every D, write the new Qs.

Most of the 92 flops are `dfrtp` (on reset, Q = 0). Four are `dfstp`: reset *sets* them to 1. Four more are `dfxtp` and have no reset pin at all.

There's also that floating net `n0550`. For now force it to 0 - since it doesn't influence success anyway, we'll investigate it at the end.


In [10]:
from pathlib import Path

from tools.netlist_ir import Design
from tools.circuit_eval import CircuitEvaluator
from tools.play import Play, bits_at, show_grid

ListOfNets = Path('generated/puzzle.json')
if not ListOfNets.exists():
    ListOfNets = Path('artifacts/netlists/puzzle.json')

GENERATED = Path('generated')
GENERATED.mkdir(exist_ok=True)

design = Design.load(ListOfNets)
sim = Play(design)  # reset / tick / scan / replay, with n0550 forced to 0
print('loaded', ListOfNets)
print('flip-flops:', len(sim.state_net))
print('success is driven by:', [t.name for t in design.resolve_net('success').drivers])


loaded generated/puzzle.json
flip-flops: 92
success is driven by: ['U28_dfrtp_2.Q']


In [11]:
q = sim.q
reset_state = sim.reset_state
tick = sim.tick
scan = sim.scan
replay_attempt = sim.replay_attempt
state_net = sim.state_net

for index, bits in enumerate(attempts, 1):
    byte_values, success_values = replay_attempt(bits)
    text = bytes(byte_values).split(b'\0', 1)[0].decode('ascii')
    print(f'attempt {index}: {text!r}  success={any(success_values)}')


attempt 1: 'TRY AGAIN'  success=False
attempt 2: 'TRY AGAIN'  success=False


Our sanity check worked, replaying the inputs that were played in `example_inputs.vcd` produces the same output. Now we can start opening the box. We'll attempt to slowly abstract groups of flip-flops into functions we understand; hopefully after enough passes things will start to make sense.


## 6. Grouping flip-flops

### 6.1. Shift Registers

Taking inspiration from the warm-up; if `I` is a 121-bit serial stream, it probably lands in a shift register: a mux in front of a D-flop, the mux selecting between "hold Q" and "take the previous bit," repeated down a chain. We look for that wiring and find a 12-bit shift register. We'll also visualize it - draw the shift register, and its incoming and outgoing connections.

In [11]:
!dot -Tsvg artifacts/explainers/puzzle-shift-register-block.dot -o generated/puzzle-shift-register-block.svg

Number of shift registers: 1

shift_I: ShiftRegister[12]
  serial input: I
  enable:       n0005
  clock:        clk
  members:      12 mux, 12 flip-flops

Wrote generated/puzzle-shift-register-block.dot
Render with: dot -Tsvg generated/puzzle-shift-register-block.dot -o generated/puzzle-shift-register-block.svg


<img src="generated/puzzle-shift-register-block.svg" alt="shift_I and the four Q bits that leave it" width="560">

Of the 12 output bits, eight - Q1 through Q8 - do not drive anything. Four stages - 0, 9, 10, and 11 — land on two gates, U360_a22o and U374_a221o. Looking up their definitions, these gates perform:

a22o  : X = (A1 & A2) OR (B1 & B2)

a221o : X = (A1 & A2) OR (B1 & B2) OR C1

We'll come back to these later. A quick note, `n0005` is not the top-level `enable` pin, so some other logic decides when this should be enabled.

### 6.2. Finite State Machines

For the remaining 80 flip-flops, we'll group them into *strongly connected components* (SCCs).

For each flop `B`, walk backwards from its D pin through (non-memory) logic gates until you hit another flop's output Q. If flop `A`'s Q shows up in that cone, then `A`'s current value can change what `B` becomes on the next clock. Draw an edge `A → B`.

A **strongly connected component** (SCC) is a set of nodes where you can follow edges from every member to every other member. All components participate in predicting the next bit of all other components - a *finite state machine*. 

Some simple examples are:

- 1-bit FSM: a flag that holds itself : `1 → 1 → 1...` or `0 → 0 → 0 ...`

- 2-bit FSM: a saturating counter : `00 → 01 → 10 → 11 → 11 → 11 ...`

We first extract all the SCCs, and will later probe what each of them do.

In [12]:
from collections import Counter
from tools.state_graph import classify_two_bit_sccs, strongly_connected_components

sr = design.strict_shift_registers().shift_registers[0]
seq = {
    inst.name: inst
    for inst in design.instances.values()
    if inst.sequential
}
shift_ffs = {stage.flip_flop for stage in sr.stages}

deps = {
    name: set(design.backward_cone(inst.pins['D'].net).state_boundaries)
    for name, inst in seq.items()
}
sccs = strongly_connected_components(deps)
thin, fat, reused, singletons = classify_two_bit_sccs(design, sccs, hide=shift_ffs)
loners = singletons  # later cells still use this name


print('SCC size histogram:')
for size, count in sorted(Counter(len(component) for component in sccs).items()):
    print(f'  {count} of size {size}')


SCC size histogram:
  25 of size 1
  23 of size 2
  1 of size 4
  1 of size 8
  1 of size 9


In [13]:
!dot -Tsvg artifacts/explainers/blocks-structural.dot -o generated/blocks-structural.svg

<img src="generated/blocks-structural.svg" alt="structural block overview" width="650">


That's simplified 92 memory elements into a handful of shift registers and FSMs. However, we don't yet know what any of these boxes *do*.
We'll start with the 9-bit FSM, it's quite big and doesn't even depend on the input stream `I`.

### 6.3. The 9 bit FSM

Hold `I` at 0, set `enable = 1` for some click cycles, and observe how the 9 bits evolve.

In [24]:
import importlib
import tools.helpers.display as _hub_display
importlib.reload(_hub_display)
from tools.helpers.display import show_bit_trace

COLCOUNT = ['U347_dfrtp_2', 'U354_dfrtp_2', 'U344_dfrtp_2', 'U346_dfrtp_2']  # LSB first
ROWCOUNT = ['U462_dfrtp_2', 'U466_dfrtp_2', 'U459_dfrtp_2', 'U458_dfrtp_2']
DONE = 'U342_dfrtp_2'
HUB = COLCOUNT + ROWCOUNT + [DONE]
HUB_LABELS = (
    [f'COLCOUNT[{i}]' for i in range(4)]
    + [f'ROWCOUNT[{i}]' for i in range(4)]
    + ['DONE']
)

trace = []
state = reset_state()
for t in range(126):
    trace.append([q(state, name) for name in HUB])
    state = tick(state, t < 121, False).next_state

show_bit_trace(
    trace,
    windows=[
        range(0, 11),
        range(11, 22),
        [22, 33, 44],
        range(111, 122),
        range(122, 126),
    ],
    row_labels=HUB_LABELS,
    row_groups=[4, 4, 1],
)


White is 0 and Black is 1. The 9 bits of the FSM are shown evolving with time. Its a counter!

The top four bits are a binary 0…10 counter that ticks every clock. The next four rows are the same counter, but they tick only when the first one wraps around — compare `t = 0 … 10` with `t = 11 … 21`. At `t = 121`, `U342` goes high (and remains there) and both of the 4-bit counters are cleared. 

If the 121 bits are an 11×11 board, `COLCOUNT` is counting columns and `ROWCOUNT` is counting rows. The ninth bit, `U342`, represents **done** (all 121 bits are read).

We'd mentioned previously that the shift-register's enable (net `n0005`) wasn't the enable to the circuit. Let's quickly query what drives it:

In [19]:
!python -m tools.analyze_netlist {ListOfNets} net n0005 --driver

Net:        n0005
Driver:     U351_and2b_2.X


And then check what inputs go into U351:

In [20]:
!python -m tools.analyze_netlist {ListOfNets} instance U351 --inputs

Instance:   U351_and2b_2
  A_N        <- n0324 <- U342_dfrtp_2.Q
  B          <- enable <- enable


Clearly, the A_N pin (N signifies that the input is inverted) is the output of U342, or the `done` flag, and input B is the global enable signal. So, we have:

```text
n0005 = enable AND !done
```

Very satisfying! So the shift register only moves when the main enable is on *and* the `done` bit hasn't flagged. After input bit 121, `done` sticks to 1, the 12-bit shift register freezes, and, with some digging, the instance `U26` goes high on the next clocl edge. This starts the output generation; we've previously encountered `TRY AGAIN`.

### 6.4. The 23 two-bit FSMs

<img src="generated/blocks-structural.svg" alt="structural block overview" width="650">

Two flops hold four states, so every one is probably a 2-bit counter or flag. They all read the input data stream `I` and the recently discovered column/row counters — so each is watching the board scan past and perhaps keeping some tally.

Instead of tracing circuits to infer their function, lets perturb the inputs and see their outputs. We'll (effectivey) send in one-hot encoded inputs, and see which positions being equal to 1 get a response out of each FSM. If an FSM's next state differs, that position is in its **territory**. We'll check out a few FSMs, from which the patterm emerges.

In [25]:
from tools.play import i_toggle_hits
from tools.helpers.display import show_grids, show_color_grid

thin_pairs = [info[0] for info in thin]
fat_pairs = [info[0] for info in fat]
row_pair = reused[0][0]

# Toggle I at each scan position; an FSM responds to the position where that flip changes
# its next state.
first_three = i_toggle_hits(sim, thin_pairs[:3])
show_grids(*((f'FSM #{i}', bits_at(*cells)) for i, cells in enumerate(first_three)))
for i, cells in enumerate(first_three):
    print(f'FSM #{i}: reacts to {len(cells)} cells, all in column {sorted({c % 11 for c in cells})}')

FSM #0: reacts to 11 cells, all in column [6]
FSM #1: reacts to 11 cells, all in column [2]
FSM #2: reacts to 11 cells, all in column [1]


Each responds to a particular **column** and nothing else. Eleven of the 23 FSMs are responsible for one column each. But what exactly are they responsible for?


In [26]:
# FSM #0 owns column 6. Put 0..4 stars in that column and read its two flops.
pair = sorted(thin_pairs[0])
col = next(iter({c % 11 for c in first_three[0]}))
lsb, msb = pair[0], pair[1]

print(f'column {col}')
print(f'{"Ones":<8}state')
for k in range(5):
    state = scan(bits_at(*[row * 11 + col for row in range(k)]))
    lo, hi = q(state, lsb), q(state, msb)
    print(f'{k:<8}{hi}{lo}')


column 6
Ones    state
0       00
1       01
2       10
3       11
4       11


It is counting the number of "stars" (the night sky awaits!) in that column: `00`, `01`, `10` for 0, 1, 2, then it sticks at `11` if you put a third or fourth. Any more than two stars is a bust.

There are 11 other FSMs that take inputs from the `(row, col)` counter, we repeat the same change-one-position idea to idenfity their territories.

In [27]:
from tools.helpers.display import show_color_grid

fat_terr = i_toggle_hits(sim, fat_pairs)

# Label each region by the first (lowest-index) cell it owns, then build an 11x11 map.
order = sorted(range(len(fat_pairs)), key=lambda i: min(fat_terr[i]))
label_of = {fat_i: label for label, fat_i in enumerate(order)}
region_map = [[next(label_of[i] for i, cells in enumerate(fat_terr) if r * 11 + c in cells)
               for c in range(11)] for r in range(11)]

show_color_grid([region_map[r][c] for r in range(11) for c in range(11)],
                title='Responsive regions')

A fully tiled space, with a nice "JSC" (Jane Street Capital) prominently displayed. Following the same logic, these FSMs also ensure that there are exactly 2 stars in each region.

**The 23rd FSM.** Since we haven't yet encountered something to check the star count within rows, this will probably be it. It makes sense that there's only one FSM for it - atmost one row can be "active" at a time, and it refreshes itself after every 11 cycles. We can run the same perturbation analysis and see which positions it 

In [28]:
row_hits = i_toggle_hits(sim, [row_pair])[0]
show_grids(('23rd FSM', bits_at(*row_hits)))
silent = sorted({c % 11 for c in set(range(121)) - row_hits})
print(f'reacts to {len(row_hits)} cells; silent on column {silent}')

from tools.helpers.display import show_row_wrap_table

row_state = sorted(row_pair)  # U407, U411 — MSB, LSB

def pair_bits(state):
    return ''.join(str(q(state, name)) for name in row_state)

trials = []
for k in range(5):
    bits = bits_at(*range(0, 2 * k, 2))
    trials.append((bits, pair_bits(scan(bits, clocks=10)), pair_bits(scan(bits, clocks=11))))
show_row_wrap_table(trials)

reacts to 110 cells; silent on column [10]


A particular row,FSM state whilecolumn counter = 10,FSM state after column counter resets to zero
0,00,00
1,01,00
2,10,00
3,11,00
4,11,00


Read the pair as bits (MSB `U407`, LSB `U411`). The black squares in row 1 are counted as `00`, `01`, `10`, `11`, then stuck at `11`. After the column counter wraps back to zero, the FSM is also `00`. It can thus be reused across rows because it resets when the column counter resets.

Since entering a new row wipes the count to `00`, we'd need a "fail" bit to flag when any row exceeds the star limit, and for that flag to remain high (fail) once set. Indeed, we find exactly that, the flip-flop responsible is U410.

In [29]:
!python -m tools.analyze_netlist {ListOfNets} instance U410 --inputs
print()
!python -m tools.analyze_netlist {ListOfNets} instance U412 --inputs
print()
!python -m tools.analyze_netlist {ListOfNets} instance U418 --inputs

!dot -Tsvg artifacts/explainers/bad-row-sticky.dot -o generated/bad-row-sticky.svg

Instance:   U410_dfrtp_2
  CLK        <- n0407 <- U405_clkbuf_8.X
  D          <- n0413 <- U412_a31o_2.X
  RESET_B    <- rst_n <- rst_n

Instance:   U412_a31o_2
  A1         <- n0325 <- U349_and4bb_2.X
  A2         <- n0005 <- U351_and2b_2.X
  A3         <- n0417 <- U418_mux2_1.X
  B1         <- n0414 <- U410_dfrtp_2.Q

Instance:   U418_mux2_1
  A0         <- n0418 <- U421_nand2_2.Y
  A1         <- n0419 <- U419_or2_2.X
  S          <- n0410 <- U407_dfrtp_2.Q


Plotting all the connections makes things a little clearer.

<img src="generated/bad-row-sticky.svg" alt="U410 row-failure flag and the row-count pair" width="720">

Gate `U412` performs `X = (A1 & A2 & A3) OR B1`, and note that the `B1` pin is simply the output from our row-failure flag `U410.Q`. 

The output of `U412` (and hence the input of `U410`) is then:

`U410.D` = ( (Enable Window) & row-fail ) OR `U410.Q`.

Once `U410.Q = 1`, it remains so because of this feedback loop, making the fail flag always high.

`A1` is `col == 10` and `A2` is `enable & !done`, i.e. the "Enable window" - when the circuit is accepting inputs. The chance to update the row-fail flag hence opens up only on the last position of a row (col == 10).

At the time instance when `col==10`, Mux `U418` is steered by the MSB (`U407`). Its data pins are effectively `(LSB & I)` and `LSB OR I` — so its output is "count so far, plus the 11th input (which is the current input `I`) is not 2."

`U410` only stays 0 if every row closed with exactly two stars. Pretty neat!

In summary, the 23 two-bit machines are divied up as:

- **11 count stars per column**,
- **11 count stars per region**, and
- **1 counts stars in the current row**.

Each is a counter that reads 0, 1, 2, then *bust*. The `success` net accepts a grid only when every column and region counter reads exactly two and no row ever tripped its sticky flag. There are even more conditions that need to be satisfied, as we'll see ahead.

### 6.5. The adjacency latch

The shift register `shift_I` has outputs that are accessed (tapped) by other logic gates - positions **0, 9, 10, 11** — which is basically the values shifted away from the current input `I` by **1, 10, 11, and 12** positions respectively. On an 11-position-wide grid, those four offsets are the previously-scanned neighbors of the current input: left (1), upper-right (10), directly above (11), and upper-left (12). That looks a lot like a **"no two neighoring stars"** check. We can test this too by simply putting the relevant inputs in and watching what happens.

In [ ]:
# The 1-bit fail flag whose inputs are the four taps + the column counter.
tap_ffs = {sr.stages[i].flip_flop for i in (0, 9, 10, 11)}
col_and_done = set(COLCOUNT) | {DONE}
ADJ = next(name for name in loners
           if tap_ffs <= (deps[name] - {name}) <= (tap_ffs | col_and_done))
print('adjacency latch:', ADJ.split('_')[0], '\n')

tests = {
    'lone star': bits_at(0),
    'star at offset 1': bits_at(0, 1),
    'star at offset 10': bits_at(1, 11),
    'star at offset 11': bits_at(0, 11),
}
for label, bits in tests.items():
    print(f'  {q(scan(bits, clocks=13), ADJ)}   {label}')

adjacency latch: U381 

  0   lone star
  1   horizontal neighbor (offset 1)
  1   diagonal across rows (offset 10)
  1   vertical neighbor (offset 11)
  0   same row, 10 apart (a wrap, NOT adjacent)


Now that that checked out, We can rechristen the flip-flop `Adj` (for adjacency checker). The final `success` requires that it stays 0.

### 6.6 Count all the stars

Eight singleton flops are still unaccounted for. They're connected in a unidirectional chain rather than a SCC. We might have a binary counter: bit *k* depends on bits `0..k` and no feedback loop. 
Find the chain by starting from the flop whose input is `done`, then proceed.

In [31]:
pop_bits = []
covered = {DONE}
remaining = set(loners) - {ADJ, BAD_ROW, 'U26_dfrtp_2', 'U27_dfrtp_2', 'U28_dfrtp_2'}
while True:
    nxt = [
        name for name in remaining
        if (deps[name] - {name}) <= covered and name not in covered
    ]
    if not nxt:
        break
    nxt.sort(key=lambda name: (len(deps[name]), name))
    pop_bits.append(nxt[0])
    covered.add(nxt[0])
    remaining.remove(nxt[0])

print('population bits LSB -> MSB:')
for i, name in enumerate(pop_bits):
    print(f'  [{i}] {name}  cone {sorted(deps[name])}')

def population(state):
    return sum(q(state, name) << i for i, name in enumerate(pop_bits))

print()
for count in [0, 1, 2, 4, 8, 22, 38]:
    bits = '1' * count + '0' * (121 - count)
    state = scan(bits)
    print(f'injected ones={count:2}  counter={population(state):2}')


population bits LSB -> MSB:
  [0] U444_dfrtp_2  cone ['U342_dfrtp_2', 'U444_dfrtp_2']
  [1] U442_dfrtp_2  cone ['U342_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2']
  [2] U440_dfrtp_2  cone ['U342_dfrtp_2', 'U440_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2']
  [3] U438_dfrtp_2  cone ['U342_dfrtp_2', 'U438_dfrtp_2', 'U440_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2']
  [4] U451_dfrtp_2  cone ['U342_dfrtp_2', 'U438_dfrtp_2', 'U440_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2', 'U451_dfrtp_2']
  [5] U450_dfrtp_2  cone ['U342_dfrtp_2', 'U438_dfrtp_2', 'U440_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2', 'U450_dfrtp_2', 'U451_dfrtp_2']
  [6] U452_dfrtp_2  cone ['U342_dfrtp_2', 'U438_dfrtp_2', 'U440_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2', 'U450_dfrtp_2', 'U451_dfrtp_2', 'U452_dfrtp_2']
  [7] U445_dfrtp_2  cone ['U342_dfrtp_2', 'U438_dfrtp_2', 'U440_dfrtp_2', 'U442_dfrtp_2', 'U444_dfrtp_2', 'U445_dfrtp_2', 'U450_dfrtp_2', 'U451_dfrtp_2', 'U452_dfrtp_2']

injected ones= 0  counter= 0
injected ones= 1  counter= 1
i

Okay, it simply counts the total number of 1's in our 121-bit-long input stream. The final gate whose output is the `success` pin is just a big old AND gate over all the flags and checks we've discovered. Lets work backwards from `success`

In [37]:
!python -m tools.analyze_netlist {ListOfNets} net success --drive

Net:        success
Driver:     U28_dfrtp_2.Q


In [38]:
!python -m tools.analyze_netlist {ListOfNets} instance U28 --inputs

Instance:   U28_dfrtp_2
  CLK        <- n0260 <- U71_clkbuf_8.X
  D          <- n0280 <- U33_a32o_2.X
  RESET_B    <- rst_n <- rst_n


In [39]:
!python -m tools.analyze_netlist {ListOfNets} instance U33 --inputs

Instance:   U33_a32o_2
  A1         <- n0319 <- U393_inv_2.Y
  A2         <- n0320 <- U55_and2_2.X
  A3         <- n0321 <- U54_and4b_2.X
  B1         <- success <- U28_dfrtp_2.Q
  B2         <- n0322 <- U53_nand2b_2.Y


After some more laborious gate-chasing, we have the success criteria:

```text
done after 121 input clocks
AND  total population = 22
AND  every row count = 2
AND  every column count = 2
AND  every region count = 2
AND  Adj never fired
```

Finally, we've interpreted all of our FSMs, the connecctivity graph looks like this:

In [40]:
!dot -Tsvg artifacts/explainers/blocks-named.dot -o generated/blocks-named.svg

<img src="generated/blocks-named.svg" alt="named block diagram" width="950">

## 8. Solve for inputs given constraints.

We'll simply put our custom regions into an online [Star Battle solver](https://www.noq.solutions/starbattle), and the solution pops out, which the site confirms is the unique solution.

<img src="artifacts/images/solution.jpg" width="450">

Hardcoding this input into the circuit and simulating gives:


In [ ]:
bits = (
    "00000001010"
    "10000100000"
    "00000001010"
    "10100000000"
    "00001010000"
    "00100000100"
    "00001000001"
    "01000010000"
    "00010000001"
    "00000100100"
    "01010000000"
)
byte_values, success_values = replay_attempt(bits, trailing_edges=20)
print(bytes(byte_values).split(b'\0', 1)[0].decode('ascii'))
print('success:', success_values[0])
